# A Test Step by Step

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/douglasrizzo/catsim/blob/main/notebooks/03_a_test_step_by_step.ipynb)

**For:** researchers who want to see inside one adaptive test, and integrators who use catsim's
computations in a testing application. **Level:** intermediate. You know Python; the
[introduction](https://douglasrizzo.com.br/catsim/introduction.html) covers the item response theory
this notebook needs, and
[Terms and notation](https://douglasrizzo.com.br/catsim/terms-and-notation.html) the terms it uses.

A simulation runs every test in three steps: start a session, select an item, answer it, and repeat
the last two until the test ends. You will take those steps one at a time and:

- read a session: its items, responses and estimates, and why it ended;
- see what a session does once it has ended;
- end a test because the selector has no item left;
- pass exposure rates to the selector;
- check that `simulate` runs the same steps.

catsim computes each step and keeps nothing between them. An application that gives tests keeps its
sessions, the item on screen and its exposure counts itself; the
[Embedding catsim](https://douglasrizzo.com.br/catsim/embedding-catsim.html) guide says what catsim
checks in its inputs and what the application owns. To run the notebook outside this repository,
install catsim with its plot extra, which adds matplotlib: `pip install 'catsim[plot]'`.

In [ ]:
import matplotlib.pyplot as plt
import numpy

from catsim import CatEngine, ItemBank, Session, plot, simulate
from catsim.estimation import EAPEstimator
from catsim.initialization import FixedPointInitializer
from catsim.selection import MaxInfoSelector
from catsim.stopping import MaxLengthStopper, StandardErrorStopper, StopRule

## An engine and an examinee

A `CatEngine` holds the item bank and the four strategies of a test, and never changes. This one
starts at an ability estimate of 0, gives the most informative item at the current estimate,
estimates ability by expected a posteriori (EAP), and stops once the standard error is at most
`TARGET_SE`, or after `MAX_ITEMS` items when the bank cannot reach that precision.

The examinee has a true ability of `TRUE_THETA`. Their answer to every item of the bank is drawn
before the test, correct with the probability the item's characteristic curve gives at that ability,
as `simulate` draws a generated examinee; in an application, each answer comes from a person. Three
seeds make the notebook repeat: `BANK_SEED` draws the bank, `RUN_SEED` seeds the generator every
step draws from, which here only breaks exact ties between items, and `EXAMINEE_SEED` draws the
answers.

In [ ]:
BANK_SEED = 1
RUN_SEED = 2
EXAMINEE_SEED = 3
N_ITEMS = 200
TARGET_SE = 0.4
MAX_ITEMS = 30
TRUE_THETA = 1.0

item_bank = ItemBank.generate(N_ITEMS, model="3PL", rng=BANK_SEED)
stopper = StopRule(StandardErrorStopper(TARGET_SE), MaxLengthStopper(MAX_ITEMS))
engine = CatEngine(item_bank, FixedPointInitializer(0.0), MaxInfoSelector(), EAPEstimator(), stopper)
answers = numpy.random.default_rng(EXAMINEE_SEED).random(N_ITEMS) < item_bank.icc(TRUE_THETA)
rng = numpy.random.default_rng(RUN_SEED)

## Start, select, answer

`engine.start` returns a `Session`, the record of one test as a value that never changes: the items
administered and their responses, in order; the estimate of every state of the test, each with its
standard error and the number of times the estimator evaluated its objective; the excluded items,
which the session must not be given; and the stop reason, None while the test runs.

`engine.select` proposes the next item. It returns the item's index, the session to answer it in,
and the items an exposure control declined, none here. `engine.answer` records the response and
returns a new session, with the estimate updated.

In [ ]:
session = engine.start(rng=rng)
print(session)

selection = engine.select(session, rng=rng)
item = selection.item
print(item, item_bank.names[item], selection.declined)

answered = engine.answer(selection.session, item, answers[item], rng=rng)
print(answered.administered_items, answered.responses, answered.estimates[-1])
print(session.administered_items, session.current_theta)

The new session has no items, and its one estimate is the initializer's 0, with an infinite standard
error: no response has said anything yet. The selector proposes item 16, `item_016`, the most
informative item at 0, and declines nothing, since this engine has no exposure control. The answered
session holds the item, the correct response and the EAP estimate after it, 0.28 with a standard
error of 0.93. The session `start` returned is unchanged: `answer` made a new one.

## The whole test

`run_test` repeats the two steps until `select` returns no item, which is how every test ends. The
loop then reads the session: each item given, its response and the estimate after it.

In [ ]:
def run_test(engine: CatEngine, session: Session, *, exposure_rates: numpy.ndarray | None = None) -> Session:
  """Select and answer items until the session ends, and return the ended session."""
  while True:
    selection = engine.select(session, rng=rng, exposure_rates=exposure_rates)
    session = selection.session
    if selection.item is None:
      return session
    session = engine.answer(session, selection.item, answers[selection.item], rng=rng)


session = run_test(engine, engine.start(rng=rng))
for item, response, estimate in zip(session.administered_items, session.responses, session.estimates[1:], strict=True):
  outcome = "correct" if response else "incorrect"
  print(f"{item_bank.names[item]}: {outcome:9} estimate {estimate.theta:5.2f}, standard error {estimate.se:.3f}")
print(session.stop_reason)

Each correct response raises the estimate and each incorrect one lowers it, by smaller steps as the
test goes on, and the standard error falls with every item. After the 21st item it is 0.399, at most
`TARGET_SE`, and the session ends with the stop reason `stopper`. The final estimate, 0.93, is
within one standard error of the true ability, 1.

The stopper is a `StopRule` of two criteria. Asked about a session, through the context
`engine.context` gives for it, the rule tells which of them hold. The cell also tries to take one
more step on the ended session.

In [ ]:
print(stopper.satisfied(engine.context(session)))
print(engine.select(session, rng=rng).item)
try:
  engine.answer(session, 0, answers[0], rng=rng)
except ValueError as error:
  print(error)

The precision criterion ended the test, before the 30-item cap. A session that has ended takes no
more steps: `select` returns no item, None, with the same session, and `answer` raises an error that
says why, as it does for any input it cannot accept, such as an item the session already had.

## When the selector runs out

The other stop reason, `no_items`, ends a session whose selector has no item left to offer. Here the
examinee may be given only `N_LEFT` items of the bank: `start` excludes the rest, as an application
would exclude the items an examinee saw in an earlier attempt.

In [ ]:
N_LEFT = 4

short = run_test(engine, engine.start(rng=rng, excluded_items=range(N_LEFT, N_ITEMS)))
print(short.administered_items, short.stop_reason, f"{short.current_se:.3f}")

The selector had only items 0 to 3 to offer and gave all four, each the most informative of those
left at the estimate of the moment. With no item left, `select` ended the session with the stop
reason `no_items`, its standard error still 0.792, far above the target. In a simulation, a recorded
examinee who runs out of answered items ends the same way.

## Exposure rates

A selector with an exposure cap, `r_max`, passes over the items whose exposure rate has reached it,
while other items remain. catsim keeps no exposure counts: the caller passes each item's rate to
`select`, from the counts it keeps. Here every item the first test gave has a rate of `GIVEN_RATE`,
as if that share of the earlier examinees had had it, against a cap of `R_MAX`.

In [ ]:
R_MAX = 0.3
GIVEN_RATE = 0.5

capped = CatEngine(item_bank, FixedPointInitializer(0.0), MaxInfoSelector(r_max=R_MAX), EAPEstimator(), stopper)
rates = numpy.zeros(N_ITEMS)
rates[list(session.administered_items)] = GIVEN_RATE
second = run_test(capped, capped.start(rng=rng), exposure_rates=rates)
print(set(second.administered_items) & set(session.administered_items))
print(len(second.administered_items), second.stop_reason, f"{second.current_se:.3f}")

The second test shares no item with the first: every item the first gave has a rate above the cap,
and other items remain. Those are less informative at the examinee's ability, so the second test
needs 27 items to reach the target precision, against the first test's 21. In a simulation,
`simulate` counts the exposure and passes the rates; an application computes them from counts of its
own.

## Reading a session as a figure

`plot.session_progress` draws a session from its items and estimates: the estimate after each item,
with a band of one standard error on each side, the difficulty of each item given and, when given,
the true ability.

In [ ]:
plot.session_progress(session, item_bank, true_theta=TRUE_THETA)
plt.show()
print(f"standard error after the first item: {session.estimates[1].se:.2f}, at the end: {session.current_se:.2f}")

The estimate (blue) starts at 0, rises past the true ability (dashed) with the first four correct
responses, drops at the first incorrect one, and settles near 1. The shaded band of one standard
error on each side narrows as the standard error falls from 0.93 after the first item to 0.40 at the
end.
Most item difficulties (orange) lie a little below the estimate at which the item was chosen: with
guessing, an item's information peaks a little above its difficulty.

## The same steps in a simulation

`simulate` runs every session with these three steps. Given this examinee's answers as a recorded
examinee, one row of a response matrix, it gives the same test: no strategy here draws a random
number unless two items tie exactly, and none do.

In [ ]:
result = simulate(engine, responses=[answers], rng=RUN_SEED)
print(result.sessions[0] == session)

## Where to go next

- The [Embedding catsim](https://douglasrizzo.com.br/catsim/embedding-catsim.html) guide covers
  these steps as a reference, what catsim checks in each input, and what an application owns:
  storing sessions, the item on screen, exposure counts and sessions that run at the same time.
- [04_custom_strategies.ipynb](04_custom_strategies.ipynb) writes a strategy of your own and calls
  it on a session.
- The [Running simulations](https://douglasrizzo.com.br/catsim/running-simulations.html) guide runs
  many tests at once.